## Master’s Thesis – Transkribus Export

This notebook exports already digitized magazine issues
from Transkribus using ACDH tooling.

[Documentation and where I got the code from](https://github.com/acdh-oeaw/acdh-transkribus-utils)

I used the acdh-transkribus-utils Python library (MIT License) to download and export XML from the Transkribus REST API (see https://github.com/acdh-oeaw/acdh-transkribus-utils). The original MIT license and copyright notice are preserved.



Author: Sophie Hamann
Created: 2026-01-20

In [ ]:
import os

from transkribus_utils.transkribus_utils import ACDHTranskribusUtils


tr_user = os.environ.get("TRANSKRIBUS_USER")
tr_pw = os.environ.get("TRANSKRIBUS_PASSWORD")

client = ACDHTranskribusUtils(user=tr_user, password=tr_pw)

# List all Collections

In [ ]:
collections = client.list_collections()
for x in collections[-7:]:
    print(x["colId"], x["colName"])

# 188933 bv-play
# 188991 Kasten_blau_45_11
# 190357 acdh-transkribus-utils
# 193145 palm
# 195363 Österreichische Bundesverfassung: Datenset A
# 196428 Österreichische Bundesverfassung: Datenset B
# 196429 Österreichische Bundesverfassung: Datenset C

In [ ]:
# defining working collection ID
col_id = 2204941          # heresies collection
documents = client.list_docs(col_id_1) # get all documents in collection

# List all documents from the given collection (heresies_collection)

In [ ]:
# checking what metadata fields are available in a collection
for k in documents[-3:]:
    print(k.keys())

In [ ]:
# getting all documents in the collection
documents = client.list_docs(col_id_1)
n = 0

#printing some metadata fields
for x in documents[n:]:
    print(
        x["docId"], 
        x["title"], 
        x["nrOfPages"])

# Inspect one Document from the Given Collection
This part of the code inspects a single document out of the Transkribus heresies_collection.
I am looking at document metadata, page lists, page xml and transcripts.

All Python code used in this thesis for accessing, inspecting, and exporting Transkribus documents is based exclusively on the publicly available and MIT-licensed acdh-transkribus-utils library, without introducing additional custom functionality beyond the methods provided in the original repository.
[see here](https://github.com/acdh-oeaw/acdh-transkribus-utils/blob/main/transkribus_utils/transkribus_utils.py)

In [ ]:
# defining document ID
doc_id_1 = 11509115         # heresies_01

In [ ]:
# inspecting document metadata
doc_metadata_1 = client.get_doc_md(doc_id_1, col_id)
print(doc_metadata_1)    

In [ ]:
# inspect document overview in markdown format
overview = client.get_doc_overview_md(doc_id_1, col_id_1)
print(overview["trp_return"]["md"])

In [ ]:
#inspecting full xml
fulldoc = client.get_fulldoc_md(doc_id_1, col_id_1, page_id=5)
print(fulldoc["doc_xml"])

In [ ]:
fulldoc = client.get_transcript(fulldoc)
print(fulldoc["transcript"][:10])

In [ ]:
# I want to see the xml structure of a page
from lxml import etree

print(etree.tostring(fulldoc["doc_xml"], pretty_print=True).decode())

# This is just checking whether this xml file has something in it. 
This code has been generated by Chat GPT

In [ ]:
fulldoc = client.get_transcript(fulldoc)
page_xml = fulldoc["page_xml"]
ns = {"page": "http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15"}
len(page_xml.xpath(".//page:TextLine", namespaces=ns))

In [ ]:
chars = page_xml.xpath(".//page:TextLine//page:Unicode/text()", namespaces=ns)
sum(len(c) for c in chars)

# Download METS files from Collection
The METS files exported from Transkribus serve as structural containers and reference PAGE-XML files, in which the actual transcribed text is encoded within <TextLine> and <Unicode> elements.

In [ ]:
from transkribus_utils.transkribus_utils import ACDHTranskribusUtils

COL_ID = 2204941          # heresies collection
DOC_ID = 11509115         # heresies_01

client = ACDHTranskribusUtils()

client.collection_to_mets(
    COL_ID,
    filter_by_doc_ids=[DOC_ID]
)

This code exports the METS XML for a single specified document from a Transkribus collection and stores it locally.

In [ ]:
COL_ID = 2204941          # heresies collection
DOC_ID = 11509115  

client = ACDHTranskribusUtils()

client.collection_to_mets(
    COL_ID,
    file_path="./foo",
    filter_by_doc_ids=[DOC_ID]
)

# Download PAGE-XML with the actual text (issue 1)
# DONE WITH CHAT GPT (cite correctly!)

In [ ]:
from transkribus_utils.transkribus_utils import ACDHTranskribusUtils
from lxml import etree
import os

client = ACDHTranskribusUtils()

COL_ID = 2204941
DOC_ID = 11509115  # heresies_01

In [ ]:
overview = client.get_doc_overview_md(DOC_ID, COL_ID)
pages = overview["pages"]

In [ ]:
# one file per page
os.makedirs("heresies_01_pagexml", exist_ok=True)

for p in pages:
    page_id = p["page_nr"]

    fulldoc = client.get_fulldoc_md(DOC_ID, COL_ID, page_id=page_id)
    fulldoc = client.get_transcript(fulldoc)

    page_xml = fulldoc["page_xml"]

    out_file = f"heresies_01_pagexml/page_{page_id}.xml"
    with open(out_file, "wb") as f:
        f.write(etree.tostring(page_xml, pretty_print=True))

In [ ]:
# one file per issue
from lxml import etree
import os

root = etree.Element("Document")

for p in pages:
    page_id = p["page_nr"]

    fulldoc = client.get_fulldoc_md(DOC_ID, COL_ID, page_id=page_id)
    fulldoc = client.get_transcript(fulldoc)

    page_xml = fulldoc["page_xml"]

    # import PAGE-XML under a single root
    root.append(page_xml)

# write one combined file
with open("heresies_01_pagexml.xml", "wb") as f:
    f.write(etree.tostring(root, pretty_print=True))